In [1]:
from sklearn.experimental import enable_iterative_imputer  # must import to enable IterativeImputer
import pandas as pd
import numpy as np
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import BayesianRidge, LogisticRegression
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.multiclass import OneVsRestClassifier
import joblib
import matplotlib.pyplot as plt
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split

In [2]:
train_data = pd.read_csv("train_B_text.csv", header=None)
test_data = pd.read_csv("test_B_text.csv", header=None)

In [3]:
train_data.head()

,0,1,2
0,Id,Title,Fake/Real
1,0,Begin Again trailer with Keira Knighley VIDEO\n,real
2,1,San Diegans share what brightens their day on ...,real
3,2,Gold Prices Hit Five-Week Low Below $1300 an O...,fake
4,3,Healthcare Innovation: Promising Vaccine in De...,fake


In [4]:
test_data.head()

,0,1
0,Id,Title
1,0,Rare Blue Diamond Found in African Mine\n
2,1,AC/DC's Future Uncertain Amid Retirement Rumors\n
3,2,First possible 'exomoon' spotted\n
4,3,Humanitarian Aid: Volunteers Bring Relief to D...


In [5]:
train_data.shape

(2449, 3)

In [6]:
test_data.shape

(1051, 2)

**Id column removal and ground truth**

In [7]:
# First, set proper column names (row 0 contains headers)
train_data.columns = train_data.iloc[0]
test_data.columns = test_data.iloc[0]

In [8]:
# Remove the header row
train_data = train_data[1:].reset_index(drop=True)
test_data = test_data[1:].reset_index(drop=True)

In [9]:
# Separate features and target for training data
X_train = train_data['Title']
y_train = train_data['Fake/Real']

In [10]:
# Test data features
X_test = test_data['Title']

In [11]:
print("Training data shape:", X_train.shape)
print("Training labels shape:", y_train.shape)
print("Test data shape:", X_test.shape)
print("\nLabel distribution:")
print(y_train.value_counts())

Training data shape: (2448,)
Training labels shape: (2448,)
Test data shape: (1050,)

Label distribution:
Fake/Real
real    1400
fake    1048
Name: count, dtype: int64


In [12]:
# Step 2: Text vectorization using TF-IDF

from sklearn.feature_extraction.text import TfidfVectorizer


In [13]:
# Create TF-IDF vectorizer
# max_features limits the number of features to prevent overfitting
# ngram_range=(1,2) uses both single words and word pairs
vectorizer = TfidfVectorizer(
    max_features=5000,  # Use top 5000 most important features
    ngram_range=(1, 2),  # Use unigrams and bigrams
    strip_accents='unicode',
    lowercase=True,
    stop_words='english'  # Remove common English words
)

In [14]:
# Fit on training data and transform both train and test
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

In [15]:
print("Training features shape:", X_train_vectorized.shape)
print("Test features shape:", X_test_vectorized.shape)
print("Number of features:", X_train_vectorized.shape[1])

Training features shape: (2448, 5000)
Test features shape: (1050, 5000)
Number of features: 5000


In [16]:
# Step 3: Train LogisticRegression with penalty=None

from sklearn.linear_model import LogisticRegression

In [17]:
# Create and train the model
# penalty=None as required by the task
# max_iter increased to ensure convergence
model = LogisticRegression(
    penalty=None,
    max_iter=1000,
    random_state=42
)

In [18]:
# Train the model
model.fit(X_train_vectorized, y_train)

# Make predictions on training data to check performance
y_train_pred = model.predict(X_train_vectorized)

# Calculate training accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)

In [19]:
print("Training Accuracy:", train_accuracy)
print("\nClassification Report on Training Data:")
print(classification_report(y_train, y_train_pred))

Training Accuracy: 0.9991830065359477

Classification Report on Training Data:
              precision    recall  f1-score   support

        fake       1.00      1.00      1.00      1048
        real       1.00      1.00      1.00      1400

    accuracy                           1.00      2448
   macro avg       1.00      1.00      1.00      2448
weighted avg       1.00      1.00      1.00      2448



In [20]:
# Step 4: Split data into train/validation and evaluate properly

from sklearn.model_selection import train_test_split

In [21]:
# Split the vectorized data into train (80%) and validation (20%)
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_vectorized, 
    y_train, 
    test_size=0.2, 
    random_state=42,
    stratify=y_train  # Keep the same fake/real ratio in both sets
)

print("Train set size:", X_train_split.shape[0])
print("Validation set size:", X_val_split.shape[0])

Train set size: 1958
Validation set size: 490


In [22]:
# Train model on the training split only
model = LogisticRegression(
    penalty=None,
    max_iter=1000,
    random_state=42
)

model.fit(X_train_split, y_train_split)

,penalty,None
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [23]:
# Evaluate on validation set
y_val_pred = model.predict(X_val_split)
val_accuracy = accuracy_score(y_val_split, y_val_pred)

print("\n" + "="*50)
print("VALIDATION RESULTS:")
print("="*50)
print("Validation Accuracy:", val_accuracy)
print("\nClassification Report:")
print(classification_report(y_val_split, y_val_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_val_split, y_val_pred))


VALIDATION RESULTS:
Validation Accuracy: 0.7306122448979592

Classification Report:
              precision    recall  f1-score   support

        fake       0.69      0.68      0.68       210
        real       0.76      0.77      0.77       280

    accuracy                           0.73       490
   macro avg       0.72      0.72      0.72       490
weighted avg       0.73      0.73      0.73       490


Confusion Matrix:
[[142  68]
 [ 64 216]]


In [24]:
# Step 5: Improve vectorization with different parameters

# Try a different TF-IDF configuration
vectorizer_improved = TfidfVectorizer(
    max_features=10000,  # More features
    ngram_range=(1, 3),  # Add trigrams
    min_df=2,  # Ignore very rare words
    max_df=0.95,  # Ignore very common words
    strip_accents='unicode',
    lowercase=True,
    stop_words='english',
    sublinear_tf=True  # Use sublinear scaling
)

In [25]:
# Re-vectorize all data
X_train_vec_improved = vectorizer_improved.fit_transform(X_train)
X_test_vec_improved = vectorizer_improved.transform(X_test)

In [26]:
# Split again
X_train_split2, X_val_split2, y_train_split2, y_val_split2 = train_test_split(
    X_train_vec_improved, 
    y_train, 
    test_size=0.2, 
    random_state=42,
    stratify=y_train
)

In [27]:
# Train new model
model_improved = LogisticRegression(
    penalty=None,
    max_iter=2000,
    random_state=42
)

model_improved.fit(X_train_split2, y_train_split2)

,penalty,None
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [28]:
# Step 6: Explore the vectorized data

# Look at the shape and sparsity
print("Vectorized data shape:", X_train_vectorized.shape)
print("Data type:", type(X_train_vectorized))
print("Is sparse matrix:", hasattr(X_train_vectorized, 'toarray'))
print("Sparsity: {:.2f}%".format(100 * (1 - X_train_vectorized.nnz / (X_train_vectorized.shape[0] * X_train_vectorized.shape[1]))))

Vectorized data shape: (2448, 5000)
Data type: <class 'scipy.sparse._csr.csr_matrix'>
Is sparse matrix: True
Sparsity: 99.85%


In [29]:
# Convert first example to dense array to see actual values
first_example = X_train_vectorized[0].toarray()[0]
print("\nFirst title:", X_train.iloc[0][:80])  # Show first 80 chars
print("First title label:", y_train.iloc[0])
print("\nVectorized representation shape:", first_example.shape)
print("Non-zero values:", np.count_nonzero(first_example))
print("\nFirst 20 non-zero values and their indices:")
non_zero_indices = np.nonzero(first_example)[0][:20]
for idx in non_zero_indices:
    print(f"  Feature {idx}: {first_example[idx]:.4f}")


First title: Begin Again trailer with Keira Knighley VIDEO

First title label: real

Vectorized representation shape: (5000,)
Non-zero values: 3

First 20 non-zero values and their indices:
  Feature 476: 0.6778
  Feature 4421: 0.5682
  Feature 4630: 0.4667


In [30]:
# See what actual words these features represent
print("\nWhat do these features mean?")
feature_names = vectorizer.get_feature_names_out()
print("Example feature names (words/phrases):")
for idx in non_zero_indices[:10]:
    print(f"  Feature {idx} = '{feature_names[idx]}': {first_example[idx]:.4f}")


What do these features mean?
Example feature names (words/phrases):
  Feature 476 = 'begin': 0.6778
  Feature 4421 = 'trailer': 0.5682
  Feature 4630 = 'video': 0.4667


In [31]:
submission = pd.DataFrame(
    {
        "Id": test_data['Id'],
        "Prediction": model_improved.predict(X_test_vec_improved)
    }
)
print(submission)

        Id Prediction
0        0       fake
1        1       fake
2        2       real
3        3       fake
4        4       real
...    ...        ...
1045  1045       real
1046  1046       real
1047  1047       real
1048  1048       real
1049  1049       real

[1050 rows x 2 columns]


In [36]:
submission.to_csv("submission_B_text.csv", index=False)